# Hidden Markov Model (HMM)
In a standard Markov chain, we observe the state directly at every step. In a Hidden Markov Model (HMM), the system evolves through hidden states that we cannot observe. Instead, we observe signals generated by these hidden states. These observations provide indirect information about the true state of the system.

> __Definition:__ A Hidden Markov Model is defined by the tuple $\mathcal{H} = (\mathcal{S}, \mathcal{O}, \mathbf{P}, \mathbf{E}, \pi_0)$, where:
> * **Hidden state space** $\mathcal{S}$: The set of unobservable states the system occupies
> * **Observable output space** $\mathcal{O}$: The set of signals or emissions we can measure
> * **Transition matrix** $\mathbf{P}$: Governs how the hidden states evolve over time, where $p_{ij} = P(s_{t+1} = j \mid s_t = i)$
> * **Emission matrix** $\mathbf{E}$: Controls the probability of observing output $o$ given hidden state $s$, where $e_{ij} = P(o_t = j \mid s_t = i)$
> * **Initial distribution** $\pi_0$: The probability distribution over hidden states at time $t=0$
>
> The hidden states form a standard Markov chain (governed by $\mathbf{P}$) but are not directly observable. At each time step, the current hidden state generates an observable output according to $\mathbf{E}$. We observe outputs, not states.

## Hidden States and Observations
The key difference from regular Markov chains is observability. In a regular Markov chain, the state is directly observable. In an HMM, the state sequence $s_1, s_2, \ldots$ is hidden while we only observe the emissions $o_1, o_2, \ldots$.

Applications include:
* **Speech recognition**: Acoustic signals (observations) correspond to phonemes (hidden states)
* **Biological sequences**: DNA bases (observations) correspond to gene structure elements like exons and introns (hidden states)
* **Financial markets**: Price movements (observations) correspond to market regimes like bull or bear (hidden states)
* **Sentiment analysis**: Words in text (observations) correspond to emotional states (hidden states)

The emission matrix $\mathbf{E}$ specifies the probability of each observation given a hidden state. Each row $i$ of $\mathbf{E}$ sums to 1 and represents the probability distribution over observations for state $i$. If $e_{ii}$ is high (e.g., 0.99), the observations reliably indicate the hidden state. If emission probabilities are nearly uniform, observations provide little information about the hidden state.

## Three Classic Problems
Three fundamental problems arise for HMMs:

1. **Evaluation**: Given observation sequence $\mathbf{o} = (o_1, \ldots, o_T)$ and model parameters $(\mathbf{P}, \mathbf{E}, \pi_0)$, compute the probability $P(\mathbf{o} \mid \mathcal{H})$. The **forward algorithm** solves this in polynomial time.

2. **Decoding**: Given observations $\mathbf{o}$, find the most likely hidden state sequence $\mathbf{s}^* = \arg\max_{\mathbf{s}} P(\mathbf{s} \mid \mathbf{o}, \mathcal{H})$. The **Viterbi algorithm** solves this problem.

3. **Learning**: Given observations $\mathbf{o}$ but unknown parameters, estimate $(\mathbf{P}, \mathbf{E}, \pi_0)$ to maximize $P(\mathbf{o} \mid \mathcal{H})$. The **Baum-Welch algorithm** (Expectation-Maximization) learns parameters from data.

Without dynamic programming, solving these problems would require enumerating all possible state sequences, which is exponentially expensive. Dynamic programming enables polynomial-time solutions.

Let's explore each algorithm.
___

## Forward Algorithm (Evaluation Problem)
The forward algorithm computes the probability of observing a sequence $\mathbf{o} = (o_1, o_2, \ldots, o_T)$ given the HMM parameters $\mathcal{H} = (\mathbf{P}, \mathbf{E}, \pi_0)$. This probability, $P(\mathbf{o} \mid \mathcal{H})$, tells us how well the model explains the observed data. The naive approach would enumerate all possible hidden state sequences (there are $|\mathcal{S}|^T$ of them) and sum their probabilities, which is exponentially expensive. The forward algorithm uses dynamic programming to compute this probability efficiently in $O(|\mathcal{S}|^2 T)$ time.

The key insight is the forward variable $\alpha_t(i)$, defined as the probability of observing the partial sequence $(o_1, \ldots, o_t)$ and being in state $i$ at time $t$:

$$
\begin{equation*}
\alpha_t(i) = P(o_1, o_2, \ldots, o_t, s_t = i \mid \mathcal{H})
\end{equation*}
$$

The forward variables satisfy the recursion:

$$
\begin{equation*}
\alpha_{t+1}(j) = \left[\sum_{i=1}^{|\mathcal{S}|} \alpha_t(i) \cdot p_{ij}\right] \cdot e_{j,o_{t+1}}
\end{equation*}
$$

where $e_{j,o_{t+1}}$ denotes the emission probability $P(o_{t+1} \mid s_{t+1} = j)$, the probability that state $j$ emits the observation $o_{t+1}$. This says: to compute the probability of being in state $j$ at time $t+1$ and observing $(o_1, \ldots, o_{t+1})$, we sum over all possible previous states $i$, multiply by the transition probability $p_{ij}$ and the emission probability.

#### Algorithm
Let's develop the forward algorithm for computing the observation probability $P(\mathbf{o} \mid \mathcal{H})$.

__Initialize__: Given an HMM $\mathcal{H} = (\mathcal{S}, \mathcal{O}, \mathbf{P}, \mathbf{E}, \pi_0)$ and an observation sequence $\mathbf{o} = (o_1, o_2, \ldots, o_T)$. For each state $i \in \mathcal{S}$, initialize the forward variable at time $t=1$:

$$\alpha_1(i) \gets \pi_0(i) \cdot e_{i,o_1}$$

This represents the probability of starting in state $i$ and observing $o_1$.

For $t = 1$ to $T-1$ __do__:
1. For each state $j \in \mathcal{S}$, compute the forward variable at time $t+1$:
   $$\alpha_{t+1}(j) \gets \left[\sum_{i=1}^{|\mathcal{S}|} \alpha_t(i) \cdot p_{ij}\right] \cdot e_{j,o_{t+1}}$$

__Compute Observation Probability__: Sum the forward variables at the final time step:

$$P(\mathbf{o} \mid \mathcal{H}) \gets \sum_{i=1}^{|\mathcal{S}|} \alpha_T(i)$$

This sums over all possible final states, giving the total probability of the observation sequence.

__Output__: The observation probability $P(\mathbf{o} \mid \mathcal{H})$ and the forward variables $\alpha_t(i)$ for all $t$ and $i$ (which are useful for other algorithms like Baum-Welch).

The forward algorithm is essential for model evaluation, comparing different HMMs, and forms a key component of the learning algorithm.
___

## Viterbi Algorithm (Decoding Problem)
The Viterbi algorithm solves the decoding problem: given an observation sequence $\mathbf{o} = (o_1, o_2, \ldots, o_T)$, find the most likely hidden state sequence $\mathbf{s}^* = (s_1^*, s_2^*, \ldots, s_T^*)$ that produced those observations. This is fundamentally different from the forward algorithm, which sums over all possible state sequences. Viterbi finds the single best path through the state space using dynamic programming with a max operation instead of a sum.

The key insight is the Viterbi variable $\delta_t(i)$, defined as the probability of the most likely state sequence ending in state $i$ at time $t$ that accounts for the observations $(o_1, \ldots, o_t)$:

$$
\begin{equation*}
\delta_t(i) = \max_{s_1, \ldots, s_{t-1}} P(s_1, \ldots, s_{t-1}, s_t = i, o_1, \ldots, o_t \mid \mathcal{H})
\end{equation*}
$$

The Viterbi recursion is:

$$
\begin{equation*}
\delta_{t+1}(j) = \left[\max_{i} \delta_t(i) \cdot p_{ij}\right] \cdot e_{j,o_{t+1}}
\end{equation*}
$$

where $e_{j,o_{t+1}}$ denotes the emission probability $P(o_{t+1} \mid s_{t+1} = j)$. To reconstruct the optimal path, we maintain backpointers $\psi_t(j)$ that record which previous state $i$ maximized the Viterbi variable at each step:

$$
\begin{equation*}
\psi_{t+1}(j) = \arg\max_{i} \delta_t(i) \cdot p_{ij}
\end{equation*}
$$

#### Algorithm
Let's develop the Viterbi algorithm for computing the most likely state sequence $\mathbf{s}^*$.

__Initialize__: Given an HMM $\mathcal{H} = (\mathcal{S}, \mathcal{O}, \mathbf{P}, \mathbf{E}, \pi_0)$ and an observation sequence $\mathbf{o} = (o_1, o_2, \ldots, o_T)$. For each state $i \in \mathcal{S}$, initialize:

$$\delta_1(i) \gets \pi_0(i) \cdot e_{i,o_1}$$
$$\psi_1(i) \gets 0$$

The backpointer at $t=1$ is undefined since there is no previous state.

__Forward Pass__: For $t = 1$ to $T-1$ __do__:
1. For each state $j \in \mathcal{S}$, compute:
   $$\delta_{t+1}(j) \gets \max_{i \in \mathcal{S}} \left[\delta_t(i) \cdot p_{ij}\right] \cdot e_{j,o_{t+1}}$$
   $$\psi_{t+1}(j) \gets \arg\max_{i \in \mathcal{S}} \left[\delta_t(i) \cdot p_{ij}\right]$$

__Termination__: Find the most likely final state:

$$s_T^* \gets \arg\max_{i \in \mathcal{S}} \delta_T(i)$$
$$P^* \gets \max_{i \in \mathcal{S}} \delta_T(i)$$

where $P^*$ is the probability of the most likely state sequence.

__Backtracking__: Reconstruct the optimal state sequence by following the backpointers. For $t = T-1$ down to $1$ __do__:

$$s_t^* \gets \psi_{t+1}(s_{t+1}^*)$$

__Output__: The most likely state sequence $\mathbf{s}^* = (s_1^*, s_2^*, \ldots, s_T^*)$ and its probability $P^*$.

The Viterbi algorithm is fundamental for applications like speech recognition (finding the most likely phoneme sequence), part-of-speech tagging (finding the most likely grammatical tag sequence), and gene finding (identifying the most likely sequence of gene structure elements).
___

## Baum-Welch Algorithm (Learning Problem)
The Baum-Welch algorithm solves the learning problem: given only observation sequences (and possibly the structure of the state space), estimate the HMM parameters $(\mathbf{P}, \mathbf{E}, \pi_0)$ that best explain the data. This is an instance of the Expectation-Maximization (EM) algorithm applied to HMMs. Unlike the forward and Viterbi algorithms, which assume we know the model parameters, Baum-Welch learns those parameters from data when the hidden states are unobserved.

The algorithm iteratively refines parameter estimates by alternating between two steps:
* **E-step** (Expectation): Compute the expected sufficient statistics (state occupancies and transition counts) given the current parameters and observations.
* **M-step** (Maximization): Update parameters to maximize the likelihood of the observations given these expected statistics.

The key quantities are the forward variables $\alpha_t(i)$ (from the forward algorithm), the backward variables $\beta_t(i)$, and two derived probabilities:

**Backward variable** $\beta_t(i)$: The probability of observing the future sequence $(o_{t+1}, \ldots, o_T)$ given that we are in state $i$ at time $t$:

$$
\begin{equation*}
\beta_t(i) = P(o_{t+1}, o_{t+2}, \ldots, o_T \mid s_t = i, \mathcal{H})
\end{equation*}
$$

The backward recursion is:

$$
\begin{equation*}
\beta_t(i) = \sum_{j=1}^{|\mathcal{S}|} p_{ij} \cdot e_{j,o_{t+1}} \cdot \beta_{t+1}(j)
\end{equation*}
$$

where $e_{j,o_{t+1}}$ denotes the emission probability $P(o_{t+1} \mid s_{t+1} = j)$.

**State posterior** $\gamma_t(i)$: The probability of being in state $i$ at time $t$ given the entire observation sequence:

$$
\begin{equation*}
\gamma_t(i) = P(s_t = i \mid \mathbf{o}, \mathcal{H}) = \frac{\alpha_t(i) \cdot \beta_t(i)}{\sum_{j=1}^{|\mathcal{S}|} \alpha_t(j) \cdot \beta_t(j)}
\end{equation*}
$$

**Transition posterior** $\xi_t(i,j)$: The probability of transitioning from state $i$ at time $t$ to state $j$ at time $t+1$ given the observations:

$$
\begin{equation*}
\xi_t(i,j) = P(s_t = i, s_{t+1} = j \mid \mathbf{o}, \mathcal{H}) = \frac{\alpha_t(i) \cdot p_{ij} \cdot e_{j,o_{t+1}} \cdot \beta_{t+1}(j)}{\sum_{i'=1}^{|\mathcal{S}|} \sum_{j'=1}^{|\mathcal{S}|} \alpha_t(i') \cdot p_{i'j'} \cdot e_{j',o_{t+1}} \cdot \beta_{t+1}(j')}
\end{equation*}
$$

Note that $\xi_t(i,j)$ is only defined for $t = 1, \ldots, T-1$ since it involves a transition to time $t+1$.

#### Algorithm
Let's develop the Baum-Welch algorithm for learning HMM parameters from observation sequences.

__Initialize__: Given observation sequences $\mathbf{o}^{(1)}, \mathbf{o}^{(2)}, \ldots, \mathbf{o}^{(N)}$ (where each sequence may have different length $T_n$), the number of hidden states $|\mathcal{S}|$, the number of observation symbols $|\mathcal{O}|$, convergence tolerance $\epsilon$, and maximum iterations $K_{\text{max}}$. Initialize parameters randomly or using prior knowledge:
* Transition matrix $\mathbf{P}$: Each row sums to 1
* Emission matrix $\mathbf{E}$: Each row sums to 1  
* Initial distribution $\pi_0$: Sums to 1
* Set iteration counter $k \gets 0$ and $\texttt{converged} \gets \texttt{false}$

While $\texttt{converged}$ is $\texttt{false}$ __do__:

__E-Step__ (Compute expected sufficient statistics):

For each observation sequence $\mathbf{o}^{(n)} = (o_1^{(n)}, \ldots, o_{T_n}^{(n)})$, $n = 1, \ldots, N$:

1. **Forward pass**: Compute $\alpha_t^{(n)}(i)$ for all $t = 1, \ldots, T_n$ and $i \in \mathcal{S}$ using the forward algorithm.

2. **Backward pass**: Initialize $\beta_{T_n}^{(n)}(i) \gets 1$ for all $i \in \mathcal{S}$. For $t = T_n - 1$ down to $1$:
   $$\beta_t^{(n)}(i) \gets \sum_{j=1}^{|\mathcal{S}|} p_{ij} \cdot e_{j,o_{t+1}^{(n)}} \cdot \beta_{t+1}^{(n)}(j)$$

3. **Compute posteriors**: 
   - For all $t = 1, \ldots, T_n$ and $i \in \mathcal{S}$:
     $$\gamma_t^{(n)}(i) \gets \frac{\alpha_t^{(n)}(i) \cdot \beta_t^{(n)}(i)}{\sum_{j=1}^{|\mathcal{S}|} \alpha_t^{(n)}(j) \cdot \beta_t^{(n)}(j)}$$
   - For all $t = 1, \ldots, T_n - 1$ and $i, j \in \mathcal{S}$:
     $$\xi_t^{(n)}(i,j) \gets \frac{\alpha_t^{(n)}(i) \cdot p_{ij} \cdot e_{j,o_{t+1}^{(n)}} \cdot \beta_{t+1}^{(n)}(j)}{\sum_{i'=1}^{|\mathcal{S}|} \sum_{j'=1}^{|\mathcal{S}|} \alpha_t^{(n)}(i') \cdot p_{i'j'} \cdot e_{j',o_{t+1}^{(n)}} \cdot \beta_{t+1}^{(n)}(j')}$$

__M-Step__ (Update parameters using expected statistics):

1. **Update initial distribution**:
   $$\pi_0(i) \gets \frac{1}{N} \sum_{n=1}^{N} \gamma_1^{(n)}(i)$$

2. **Update transition matrix**:
   $$p_{ij} \gets \frac{\sum_{n=1}^{N} \sum_{t=1}^{T_n - 1} \xi_t^{(n)}(i,j)}{\sum_{n=1}^{N} \sum_{t=1}^{T_n - 1} \gamma_t^{(n)}(i)}$$
   
   This is the expected number of transitions from $i$ to $j$ divided by the expected number of times in state $i$ with a subsequent time step available (i.e., $t < T_n$).

3. **Update emission matrix**: For each state $i \in \mathcal{S}$ and observation symbol $o \in \mathcal{O}$:
   $$e_{i,o} \gets \frac{\sum_{n=1}^{N} \sum_{t: o_t^{(n)}=o} \gamma_t^{(n)}(i)}{\sum_{n=1}^{N} \sum_{t=1}^{T_n} \gamma_t^{(n)}(i)}$$
   
   This is the expected number of times in state $i$ and observing symbol $o$ divided by the expected number of times in state $i$.

__Check Convergence__:
* Compute log-likelihood: $\mathcal{L}_k \gets \sum_{n=1}^{N} \log P(\mathbf{o}^{(n)} \mid \mathcal{H})$ using the forward algorithm
* If $k = 0$, set $\mathcal{L}_{-1} \gets -\infty$
* If $|\mathcal{L}_k - \mathcal{L}_{k-1}| < \epsilon$ or $k \geq K_{\text{max}}$, set $\texttt{converged} \gets \texttt{true}$
* Otherwise, update $k \gets k + 1$

__Output__: The learned parameters $(\mathbf{P}, \mathbf{E}, \pi_0)$ that (locally) maximize the likelihood of the observation sequences.

#### Convergence Properties
The Baum-Welch algorithm converges to a local maximum of the likelihood function. It may not find the global maximum; results depend on initialization. 

Each iteration increases (or leaves unchanged) the likelihood $P(\mathbf{o} \mid \mathcal{H})$ due to the EM framework. In practice, multiple random initializations are used to find the parameters with the highest likelihood.

The Baum-Welch algorithm is commonly used for training HMMs in applications like speech recognition, bioinformatics, and natural language processing.
___